In [1]:
import pandas as pd
import os  # Needed for file system operations (listing files, joining paths)
import sys # For printing error messages

# --- Core Processing Function ---
# (Accepts full input path and base output filename)
def process_posts_add_flags_deduplicate(full_input_path, output_filename_base):
    """
    Reads a CSV from a full path, removes duplicate 'post_id' rows (keeping first),
    adds 'is_new' and 'post_2013' boolean columns, and saves the result
    to the /kaggle/working/ directory.

    Args:
        full_input_path (str): Full path to the input CSV file
                               (e.g., /kaggle/input/dataset_name/file.csv).
                               Requires 'post_id', 'author_user_id',
                               'post_creation_date' columns.
        output_filename_base (str): Base filename for the output CSV
                                    (e.g., processed_file.csv). Will be saved
                                    in /kaggle/working/.

    Returns:
        bool: True if processing was successful, False otherwise.
    """
    print(f"--- Starting processing for: {os.path.basename(full_input_path)} ---")
    # Construct full output path within Kaggle's working directory
    # Note: Kaggle notebooks typically run with /kaggle/working/ as the CWD,
    # but explicitly joining is safer.
    full_output_path = os.path.join('/kaggle/working/', output_filename_base)

    try:
        # Read the CSV file
        df = pd.read_csv(full_input_path, low_memory=False)
        print(f"Read {len(df):,} initial rows.")

        # --- Column Validation ---
        required_columns = ['post_id', 'author_user_id', 'post_creation_date']
        missing_cols = [col for col in required_columns if col not in df.columns]
        if missing_cols:
            # Raise error if required columns are missing
            raise ValueError(f"Missing required columns: {', '.join(missing_cols)}")

        # --- Duplicate Post ID Check and Removal ---
        print("Checking for duplicate 'post_id' values...")
        if df.duplicated(subset=['post_id']).any():
            initial_rows = len(df)
            duplicate_count = df.duplicated(subset=['post_id']).sum()
            print(f"Found {duplicate_count:,} duplicate rows based on 'post_id'. Removing duplicates, keeping first...")
            df.drop_duplicates(subset=['post_id'], keep='first', inplace=True)
            print(f"Removed {initial_rows - len(df):,} duplicate rows. Processing {len(df):,} unique rows.")
        else:
            # Use f-string for consistent formatting
            print(f"No duplicate 'post_id' values found. Processing {len(df):,} rows.")

        # --- Type Conversion ---
        print("Converting 'post_creation_date' to datetime...")
        df['post_creation_date'] = pd.to_datetime(df['post_creation_date'], errors='coerce')

        # --- Feature Engineering ---
        print("Sorting data to identify first posts...")
        df_sorted = df.sort_values(by=['author_user_id', 'post_creation_date'],
                                   ascending=True,
                                   na_position='last')

        print("Calculating 'is_new' column (True if first post by author)...")
        is_new_series = ~df_sorted.duplicated(subset=['author_user_id'], keep='first')
        df['is_new'] = is_new_series.reindex(df.index)

        print("Calculating 'post_2013' column (True if post date > 2013)...")
        df['post_2013'] = df['post_creation_date'].dt.year >= 2014

        # --- Save Output ---
        print(f"Saving processed data ({len(df):,} rows) to: {full_output_path}")
        df.to_csv(full_output_path, index=False)
        print(f"--- Successfully finished processing: {os.path.basename(full_input_path)} ---")
        return True # Indicate success

    except FileNotFoundError:
        # Use f-string for error message consistency
        print(f"❌ ERROR: Input file not found at '{full_input_path}'", file=sys.stderr)
    except ValueError as ve:
        print(f"❌ ERROR: Data validation failed for {os.path.basename(full_input_path)} - {ve}", file=sys.stderr)
    except KeyError as ke:
         print(f"❌ ERROR: Missing expected column in {os.path.basename(full_input_path)} - {ke}", file=sys.stderr)
    except Exception as e:
        # Catch any other unexpected errors during processing for a specific file
        print(f"❌ ERROR: An unexpected error occurred processing {os.path.basename(full_input_path)} - {e}", file=sys.stderr)

    return False # Indicate failure if any exception occurred

# ==============================================================================
# --- Iteration Logic ---
# ==============================================================================

# <<< --- Configuration --- >>>
# <<< CHANGE THIS to the exact name of your Kaggle dataset directory >>>
# <<< Example: INPUT_DATASET_NAME = 'msrdb2'                      >>>
INPUT_DATASET_NAME = 'msrdb-author'
# <<< --- End Configuration --- >>>

# Construct the base directory path where Kaggle mounts the input dataset
input_base_dir = f'/kaggle/input/{INPUT_DATASET_NAME}'

print(f"\n======= Starting Iterative Processing for Dataset: {INPUT_DATASET_NAME} =======")
print(f"Looking for CSV files in directory: {input_base_dir}")

# Check if the dataset directory actually exists
if not os.path.isdir(input_base_dir):
    print(f"❌ FATAL ERROR: Input dataset directory not found at '{input_base_dir}'", file=sys.stderr)
    print("Please ensure the dataset name is correct and the dataset is added to your Kaggle environment.")
else:
    # List all items (files and directories) in the base directory
    try:
        all_items_in_dir = os.listdir(input_base_dir)
    except OSError as e:
        print(f"❌ FATAL ERROR: Could not list directory contents for '{input_base_dir}' - {e}", file=sys.stderr)
        all_items_in_dir = [] # Prevent further errors

    # Filter for files that end with .csv (case-insensitive)
    csv_files = [
        f for f in all_items_in_dir
        if f.lower().endswith('.csv') and os.path.isfile(os.path.join(input_base_dir, f))
    ]

    if not csv_files:
        print(f"No CSV files found in '{input_base_dir}'. Nothing to process.")
    else:
        print(f"Found {len(csv_files)} CSV files to process.") # List removed for brevity if many files: {csv_files}")

        processed_count = 0
        failed_files = []

        # Loop through each identified CSV file
        for i, filename in enumerate(csv_files):
            print(f"\n>>> Processing file {i+1}/{len(csv_files)}: {filename} <<<")

            # Construct the full path to the input CSV file
            full_input_path = os.path.join(input_base_dir, filename)

            # Construct a unique output filename, e.g., prefix with 'processed_'
            # Basic sanitization just in case
            sanitized_filename = filename.replace('/', '_').replace('\\', '_')
            output_filename = f'processed_{sanitized_filename}'

            # Call the main processing function for the current file
            # The function now returns True for success, False for failure
            success = process_posts_add_flags_deduplicate(
                full_input_path=full_input_path,
                output_filename_base=output_filename
            )

            # Track successes and failures
            if success:
                processed_count += 1
            else:
                failed_files.append(filename)

        # --- Summary Report ---
        print(f"\n======= Iterative Processing Complete =======")
        print(f"Successfully processed {processed_count} out of {len(csv_files)} CSV files.")
        if failed_files:
            print(f"Failed to process {len(failed_files)} files:")
            # Sort failed files list for consistent reporting
            for failed_file in sorted(failed_files):
                print(f"  - {failed_file}")
        print(f"Check the '/kaggle/working/' directory for the output files (prefixed with 'processed_').")


======= Starting Iterative Processing for Dataset: msrdb-author =======
Looking for CSV files in directory: /kaggle/input/msrdb-author
Found 79 CSV files to process.

>>> Processing file 1/79: updated_authors_oceanbase_posts_with_comments_answers.csv <<<
--- Starting processing for: updated_authors_oceanbase_posts_with_comments_answers.csv ---
Read 9 initial rows.
Checking for duplicate 'post_id' values...
No duplicate 'post_id' values found. Processing 9 rows.
Converting 'post_creation_date' to datetime...
Sorting data to identify first posts...
Calculating 'is_new' column (True if first post by author)...
Calculating 'post_2013' column (True if post date > 2013)...
Saving processed data (9 rows) to: /kaggle/working/processed_updated_authors_oceanbase_posts_with_comments_answers.csv
--- Successfully finished processing: updated_authors_oceanbase_posts_with_comments_answers.csv ---

>>> Processing file 2/79: updated_authors_hypertable_posts_with_comments_answers.csv <<<
--- Starting p

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


Read 2,790 initial rows.
Checking for duplicate 'post_id' values...
Found 133 duplicate rows based on 'post_id'. Removing duplicates, keeping first...
Removed 133 duplicate rows. Processing 2,657 unique rows.
Converting 'post_creation_date' to datetime...
Sorting data to identify first posts...
Calculating 'is_new' column (True if first post by author)...
Calculating 'post_2013' column (True if post date > 2013)...
Saving processed data (2,657 rows) to: /kaggle/working/processed_updated_authors_rexx_posts_with_comments_answers (1).csv
--- Successfully finished processing: updated_authors_rexx_posts_with_comments_answers (1).csv ---

>>> Processing file 4/79: updated_authors_rexx_posts_with_comments_answers.csv <<<
--- Starting processing for: updated_authors_rexx_posts_with_comments_answers.csv ---
Read 2,790 initial rows.
Checking for duplicate 'post_id' values...
Found 133 duplicate rows based on 'post_id'. Removing duplicates, keeping first...
Removed 133 duplicate rows. Processing 

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


Read 3,238 initial rows.
Checking for duplicate 'post_id' values...
Found 33 duplicate rows based on 'post_id'. Removing duplicates, keeping first...
Removed 33 duplicate rows. Processing 3,205 unique rows.
Converting 'post_creation_date' to datetime...
Sorting data to identify first posts...
Calculating 'is_new' column (True if first post by author)...
Calculating 'post_2013' column (True if post date > 2013)...
Saving processed data (3,205 rows) to: /kaggle/working/processed_updated_authors_ibm db2_posts_with_comments_answers.csv
--- Successfully finished processing: updated_authors_ibm db2_posts_with_comments_answers.csv ---

>>> Processing file 6/79: updated_authors_couchdb_posts_with_comments_answers.csv <<<
--- Starting processing for: updated_authors_couchdb_posts_with_comments_answers.csv ---


...

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


--- Successfully finished processing: updated_authors_ansible_posts_with_comments_answers.csv ---

>>> Processing file 78/79: updated_authors_etcd_posts_with_comments_answers.csv <<<
--- Starting processing for: updated_authors_etcd_posts_with_comments_answers.csv ---
Read 18,588 initial rows.
Checking for duplicate 'post_id' values...
Found 857 duplicate rows based on 'post_id'. Removing duplicates, keeping first...
Removed 857 duplicate rows. Processing 17,731 unique rows.
Converting 'post_creation_date' to datetime...
Sorting data to identify first posts...
Calculating 'is_new' column (True if first post by author)...
Calculating 'post_2013' column (True if post date > 2013)...
Saving processed data (17,731 rows) to: /kaggle/working/processed_updated_authors_etcd_posts_with_comments_answers.csv


/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


--- Successfully finished processing: updated_authors_etcd_posts_with_comments_answers.csv ---

>>> Processing file 79/79: updated_authors_actordb_posts_with_comments_answers.csv <<<
--- Starting processing for: updated_authors_actordb_posts_with_comments_answers.csv ---
Read 62 initial rows.
Checking for duplicate 'post_id' values...
Found 1 duplicate rows based on 'post_id'. Removing duplicates, keeping first...
Removed 1 duplicate rows. Processing 61 unique rows.
Converting 'post_creation_date' to datetime...
Sorting data to identify first posts...
Calculating 'is_new' column (True if first post by author)...
Calculating 'post_2013' column (True if post date > 2013)...
Saving processed data (61 rows) to: /kaggle/working/processed_updated_authors_actordb_posts_with_comments_answers.csv
--- Successfully finished processing: updated_authors_actordb_posts_with_comments_answers.csv ---

======= Iterative Processing Complete =======
Successfully processed 79 out of 79 CSV files.
Check the

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
